In [ ]:
# notebooks start in research/, hop up to project root
import os
from pathlib import Path

if Path.cwd().name == "research":
    os.chdir("..")

# notebooks don't run main.py, so configure logging here
from doctalk.logger import setup_logging
setup_logging()

%load_ext autoreload
%autoreload 2

# read .env into the environment so langchain-google-genai
# can pick up GOOGLE_API_KEY on its own
from dotenv import load_dotenv
load_dotenv()

# check the key loaded WITHOUT printing it
print("key loaded:", os.environ.get("GOOGLE_API_KEY") is not None)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vector = embeddings.embed_query("How many layers does the encoder have?")

print("type:", type(vector))
print("length:", len(vector))
print("first 5 numbers:", vector[:5])

In [ ]:
import numpy as np

a = embeddings.embed_query("The encoder is a stack of 6 identical layers")
b = embeddings.embed_query("How many layers does the encoder have?")
c = embeddings.embed_query("Training took 3.5 days on eight GPUs")

def similarity(x, y):
    # cosine similarity: 1.0 = identical meaning, 0 = unrelated
    x, y = np.array(x), np.array(y)
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))

print("related  :", round(similarity(a, b), 4))
print("unrelated:", round(similarity(a, c), 4))

In [9]:
# pull in the ingestion pipeline so we have real chunks to store
from doctalk.pipelines.stage_01_ingestion import IngestionPipeline
from doctalk.config.configuration import ConfigurationManager
from doctalk.components.vector_store import VectorStore

# 1. get the 52 chunks from the pdf (same as notebook 01)
chunks = IngestionPipeline().run(Path("eval/test_document.pdf"))

# 2. build the vector store for a fake session id
vs_config = ConfigurationManager().get_vector_store_config()
vector_store = VectorStore(config=vs_config)

store = vector_store.build(chunks=chunks, session_id="test_session")

print("built store for test_session")

[2026-07-29 14:22:12,188] INFO - common - yaml file loaded: config\config.yaml
[2026-07-29 14:22:12,207] INFO - common - yaml file loaded: params.yaml
[2026-07-29 14:22:12,210] INFO - common - created directory at: artifacts/data_ingestion
[2026-07-29 14:22:13,347] INFO - data_ingestion - loaded 15 pages from test_document.pdf
[2026-07-29 14:22:13,358] INFO - data_ingestion - split into 52 chunks
[2026-07-29 14:22:13,358] INFO - common - yaml file loaded: config\config.yaml
[2026-07-29 14:22:13,358] INFO - common - yaml file loaded: params.yaml
[2026-07-29 14:22:13,363] INFO - common - created directory at: artifacts/sessions
[2026-07-29 14:22:16,746] INFO - _client - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
[2026-07-29 14:22:18,644] INFO - _client - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
[2026-07-29 14

In [10]:
question = "How many layers does the encoder have?"

results = store.similarity_search_with_score(question, k=8)

for i, (doc, score) in enumerate(results, start=1):
    print(f"--- result {i} | distance {round(score, 4)} | page {doc.metadata['page']} ---")
    print(doc.page_content[:200])
    print()

[2026-07-29 14:26:32,104] INFO - _client - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents "HTTP/1.1 200 OK"
--- result 1 | distance 0.4257 | page 3 ---
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, 

--- result 2 | distance 0.4257 | page 3 ---
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, 

--- result 3 | distance 0.4644 | page 3 ---
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder: The decoder is also composed of a st

--- result 4 | distance 0.4644 | page 3 ---
itself. To facilitate these residual conne

In [ ]:
# load a DIFFERENT session that was never built -> should be empty
empty_store = vector_store.load(session_id="nonexistent_session")
empty_results = empty_store.similarity_search("encoder layers", k=4)

print("chunks in a session that was never built:", len(empty_results))

In [ ]:
import os
print(os.listdir("artifacts/sessions"))